# Xenium Pancreas Cell Annotation

This notebook creates transparent, reproducible Tier_A/Tier_B annotations for the four 10x Xenium pancreas examples.

Important design choice:

- Each sample is annotated independently using all genes/probes available in that sample after minimal QC filtering.
- We first run PCA/UMAP/Leiden, then annotate Leiden clusters using marker programs and marker expression.
- The output includes QC plots and a cluster-review table so labels can be curated before downstream pseudotime.

SingleR/reference annotation can be added later, but this notebook does not assume an R/SingleR installation. The current default is the standard spatial transcriptomics/scRNA-seq pattern: unsupervised clustering plus marker validation.


In [ ]:

%matplotlib inline

import os
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")
os.environ.setdefault("NUMBA_CACHE_DIR", "/private/tmp/numba")

from pathlib import Path
import gc
import json
import importlib.util
import tarfile
import warnings
from io import StringIO

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import sparse

plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 8
sns.set_style("white")

ROOT = Path("/Users/shihongwu/SpatioEv")
DATA_ROOT = Path("/Volumes/Shihong_5/for_spatioev/pancreas_Xenium_example_data_from_10X")
OUTPUT_DIR = ROOT / "data" / "xenium_pancreas_10x"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_CONFIGS = [
    {
        "sample_id": "pdac_pancreas_v1",
        "display_name": "Human Pancreas FFPE",
        "disease_group": "PDAC",
        "panel_hint": "Human Multi-Tissue and Cancer",
        "outs_path": DATA_ROOT / "Xenium_V1_human_Pancreas_FFPE_outs",
    },
    {
        "sample_id": "pdac_io_v1",
        "display_name": "Human Ductal Adenocarcinoma FFPE",
        "disease_group": "PDAC",
        "panel_hint": "Human Immuno-Oncology",
        "outs_path": DATA_ROOT / "Xenium_V1_Human_Ductal_Adenocarcinoma_FFPE_outs",
    },
    {
        "sample_id": "pdac_addon_v1",
        "display_name": "hPancreas Cancer Add-on FFPE",
        "disease_group": "PDAC",
        "panel_hint": "Human Multi-Tissue + Add-on",
        "outs_path": DATA_ROOT / "Xenium_V1_hPancreas_Cancer_Add_on_FFPE_outs",
    },
    {
        "sample_id": "normal_nondiseased_v1",
        "display_name": "hPancreas nondiseased section",
        "disease_group": "NormalPancreas",
        "panel_hint": "Human Multi-Tissue and Cancer",
        "outs_path": DATA_ROOT / "Xenium_V1_hPancreas_nondiseased_section_outs",
    },
]

def package_available(name):
    return importlib.util.find_spec(name) is not None

def save_df(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix == ".pkl":
        df.to_pickle(path)
    else:
        df.to_csv(path, index=index)

def load_df(path):
    path = Path(path)
    if path.suffix == ".pkl":
        return pd.read_pickle(path)
    return pd.read_csv(path)

def present_columns(df, cols):
    return [c for c in cols if c in df.columns]

def make_sparse_safe_copy(X):
    return X.copy() if sparse.issparse(X) else np.asarray(X).copy()


In [ ]:

ANNOTATED_DIR = OUTPUT_DIR / "annotated_h5ad"
ANNOTATED_DIR.mkdir(exist_ok=True)
ANNOTATION_QC_DIR = OUTPUT_DIR / "annotation_qc"
ANNOTATION_QC_DIR.mkdir(exist_ok=True)

GENE_SETS = {
    "ductal_epithelial": ["EPCAM", "KRT7", "KRT8", "KRT18", "KRT19", "SOX9", "MUC1", "MUC5AC", "TFF1", "TFF2", "TFF3", "CEACAM6", "AGR2", "AGR3", "EGFR", "ERBB2", "CFTR", "FXYD2", "TM4SF4", "PROX1", "EHF"],
    # Keep duodenum strict. REG4/DMBT1/TFF3/TMPRSS2 can be PDAC/PanIN or
    # intestinalized ductal programs, so they are scored separately below.
    "duodenum_epithelial": ["CDX2", "MUC2", "KRT20", "VIL1", "FABP1", "FABP2", "ALPI", "APOA1", "APOA4", "SI"],
    "acinar_epithelial": ["AMY2A", "PRSS1", "PRSS2", "CPA1", "CPA2", "CTRB1", "CTRB2", "REG1A", "REG1B", "AQP8", "GATM", "ANPEP", "KLK1", "KLK11", "PNLIP", "CLPS"],
    "islet_endocrine": ["INS", "GCG", "SST", "PPY", "IAPP", "CHGA", "CHGB"],
    "fibroblast_stellate": ["ACTA2", "PDGFRA", "FAP", "THY1", "PDPN", "DCN", "LUM", "COL1A1", "COL1A2", "VIM", "TAGLN"],
    "endothelial": ["PECAM1", "VWF", "KDR", "CDH5", "SOX17", "ACKR1", "ADGRL4", "PLVAP", "FLT1", "CLDN5", "ESAM", "ENG", "EMCN", "CD34", "ECSCR", "CLEC14A", "AQP1", "SPARCL1", "IGFBP7"],
    "t_cell": ["PTPRC", "CD3D", "CD3E", "CD2", "CD247", "TRAC", "CD4", "CD8A", "CD8B"],
    "cytotoxic_t_nk": ["NKG7", "GNLY", "GZMA", "GZMB", "GZMK", "PRF1", "KLRD1", "KLRC1"],
    "treg_checkpoint": ["FOXP3", "IL2RA", "CTLA4", "PDCD1", "LAG3", "HAVCR2"],
    "b_cell": ["CD19", "MS4A1", "CD79A", "CD79B", "BANK1", "TCL1A", "IGHM", "IGKC"],
    "plasma_cell": ["MZB1", "JCHAIN", "SDC1", "XBP1", "DERL3", "TNFRSF17", "PRDM1", "IGHG1", "IGHG2", "IGHG3", "IGHG4", "IGHA1", "IGHA2", "IGKC"],
    "myeloid": ["LST1", "LYZ", "CD14", "CD68", "AIF1", "TYROBP", "FCGR3A", "FCGR1A", "FCGR2A", "C1QA", "C1QB", "C1QC", "CD163", "MPEG1", "CSF1R", "APOE", "CTSD", "PLA2G7", "S100A8", "S100A9", "CXCR1", "CXCR2", "ITGAX", "FGR", "LILRA5", "MCEMP1", "IL1B", "IL1A"],
    "mast_cell": ["KIT", "CPA3", "MS4A2", "GATA2", "HPGDS"],
    "proliferation": ["MKI67", "UBE2C", "TOP2A", "CENPF", "CDK1"],
    "panin_mucin_remodeling": ["MUC5AC", "TFF1", "TFF2", "TFF3", "CEACAM6", "AGR2", "AGR3", "DMBT1", "REG4", "GPX2"],
    "intestinal_like_ductal_remodeling": ["CDX2", "REG4", "DMBT1", "TMPRSS2", "GPX2", "TFF3"],
}

BROAD_SCORE_TO_TIER_A = {
    "ductal_epithelial_score": "pancreatic ductal epithelium",
    "duodenum_epithelial_score": "Duodenum epithelial",
    "acinar_epithelial_score": "pancreatic acinar epithelium",
    "islet_endocrine_score": "Islets",
    "fibroblast_stellate_score": "Fibroblasts",
    "endothelial_score": "Endothelial cells",
    "t_cell_score": "T cells",
    "b_cell_score": "B lineage",
    "plasma_cell_score": "B lineage",
    "myeloid_score": "Myeloid cells",
    "mast_cell_score": "Mast cells",
}

MARKER_EXPORT_GENES = sorted(set().union(*GENE_SETS.values()))
VALIDATION_MARKERS = [
    "EPCAM", "KRT7", "KRT19", "SOX9", "MUC1", "MUC5AC", "TFF1", "TFF2", "TFF3", "CEACAM6", "AGR2", "AGR3", "MKI67", "TOP2A",
    "CFTR", "FXYD2", "TM4SF4", "PROX1", "EHF", "CDX2", "REG4", "DMBT1", "TMPRSS2", "GPX2", "MUC2", "KRT20", "VIL1", "FABP1", "FABP2", "ALPI",
    "AMY2A", "PRSS1", "CPA1", "REG1A", "AQP8", "GATM", "ANPEP", "KLK1", "KLK11", "INS", "GCG", "SST", "PPY",
    "ACTA2", "PDGFRA", "FAP", "THY1", "PDPN", "DCN", "LUM", "COL1A1", "VIM",
    "PECAM1", "VWF", "KDR", "PLVAP", "FLT1", "SPARCL1", "IGFBP7", "RGS5", "SOX17", "CD34",
    "PTPRC", "CD3D", "CD3E", "CD4", "CD8A", "FOXP3", "GZMB", "NKG7",
    "CD19", "MS4A1", "CD79A", "CD79B", "BANK1", "MZB1", "JCHAIN", "SDC1", "TNFRSF17", "IGHM", "IGKC", "IGHG1", "IGHA1",
    "LST1", "LYZ", "CD68", "AIF1", "CD14", "FCGR3A", "FCGR2A", "CD163", "MPEG1", "CSF1R", "S100A9", "CXCR2", "ITGAX", "KIT", "CPA3",
]
DOTPLOT_MARKERS = [
    "EPCAM", "KRT7", "SOX9", "CFTR", "FXYD2", "TM4SF4", "MUC5AC", "TFF2", "TFF3", "CEACAM6", "AGR3",
    "CDX2", "REG4", "DMBT1", "TMPRSS2", "GPX2", "MUC2", "KRT20", "VIL1", "FABP1", "FABP2", "ALPI",
    "AMY2A", "AQP8", "GATM", "ANPEP", "KLK11", "INS", "GCG", "SST", "CHGA",
    "ACTA2", "PDGFRA", "FAP", "THY1", "PDPN", "DCN", "LUM", "PECAM1", "VWF", "KDR", "PLVAP", "FLT1", "SPARCL1", "IGFBP7", "RGS5", "SOX17", "CD34",
    "PTPRC", "CD3D", "CD3E", "CD4", "CD8A", "FOXP3", "GZMB", "NKG7",
    "CD19", "MS4A1", "CD79A", "BANK1", "MZB1", "JCHAIN", "SDC1", "TNFRSF17", "IGKC", "IGHM",
    "LST1", "LYZ", "CD68", "AIF1", "CD14", "FCGR2A", "CD163", "MPEG1", "CSF1R", "S100A9", "CXCR2", "ITGAX", "KIT", "CPA3", "MKI67", "TOP2A",
]
TIER_A_PALETTE = {
    "pancreatic ductal epithelium": "#1f78b4",
    "Duodenum epithelial": "#8dd3c7",
    "Mucosa gland": "#80cdc1",
    "Submucosa": "#8c6d31",
    "pancreatic acinar epithelium": "#33a02c",
    "Islets": "#ff7f00",
    "Fibroblasts": "#e31a1c",
    "Endothelial cells": "#6a3d9a",
    "T cells": "#b15928",
    "B lineage": "#a6cee3",
    "Myeloid cells": "#cab2d6",
    "Mast cells": "#fb9a99",
    "Unknown": "#bdbdbd",
}
TIER_B_BASE_PALETTE = {
    "ductal/tumor epithelial": "#1f78b4",
    "proliferative ductal/tumor epithelial": "#08519c",
    "mucin/PanIN-like ductal epithelial": "#4292c6",
    "duodenum/intestinal epithelial": "#8dd3c7",
    "mucosa gland": "#80cdc1",
    "submucosa": "#8c6d31",
    "intestinal-like/PanIN-like ductal epithelial": "#2171b5",
    "acinar epithelial": "#33a02c",
    "endocrine/islet": "#ff7f00",
    "activated fibroblast/stellate": "#e31a1c",
    "fibroblast/stellate": "#fb6a4a",
    "endothelial": "#6a3d9a",
    "endothelial/perivascular": "#9e9ac8",
    "Tregs": "#b15928",
    "CD8 T cells": "#d95f02",
    "CD4/other T cells": "#fdbf6f",
    "B cells": "#a6cee3",
    "plasma-like B lineage": "#1f78b4",
    "myeloid/macrophage": "#cab2d6",
    "inflammatory myeloid": "#756bb1",
    "mast cell": "#fb9a99",
    "Unknown": "#bdbdbd",
}
LEIDEN_KEY = "leiden_annotation"
XENIUM_GRAPHCLUST_KEY = "xenium_graphclust"
XENIUM_KMEANS10_KEY = "xenium_kmeans_10"
ANNOTATION_CLUSTER_PREFERENCE = [XENIUM_GRAPHCLUST_KEY, LEIDEN_KEY]
CLUSTER_REVIEW_PATH = ANNOTATION_QC_DIR / "xenium_cluster_annotation_review.csv"
LEIDEN_RESOLUTION = 0.8
ANNOTATION_VERSION = "cluster_full_panel_v9_xenium_graphclust_io_mucosa_submucosa_k24"
CURATED_CLUSTER_LABEL_OVERRIDES = {
    ("pdac_io_v1", XENIUM_GRAPHCLUST_KEY, "2"): {
        "final_Tier_A": "Mucosa gland",
        "final_Tier_B": "mucosa gland",
        "notes": "Manual Xenium Explorer review: 10x graphclust 2 is mucosa gland, not pancreatic ductal epithelium.",
    },
    ("pdac_io_v1", XENIUM_GRAPHCLUST_KEY, "17"): {
        "final_Tier_A": "Submucosa",
        "final_Tier_B": "submucosa",
        "notes": "Manual Xenium Explorer review: 10x graphclust 17 is submucosa, not pancreatic ductal epithelium.",
    },
}

STRICT_DUODENUM_MARKERS = ["CDX2", "MUC2", "KRT20", "VIL1", "FABP1", "FABP2", "ALPI", "APOA1", "APOA4", "SI"]
NON_CDX2_DUODENUM_MARKERS = ["MUC2", "KRT20", "VIL1", "FABP1", "FABP2", "ALPI", "APOA1", "APOA4", "SI"]
INTESTINAL_LIKE_DUCTAL_MARKERS = ["CDX2", "REG4", "DMBT1", "TMPRSS2", "GPX2", "TFF3"]
EPITHELIAL_REFINEMENT_REQUIRED_GENES = ["EPCAM"]
EPITHELIAL_REFINEMENT_CANDIDATE_TIER_A = [
    "pancreatic ductal epithelium",
    "pancreatic acinar epithelium",
    "Islets",
    "Duodenum epithelial",
]
EPITHELIAL_REFINEMENT_K = 24
EPITHELIAL_REFINEMENT_MIN_CELLS = 1000

TOP_MARKER_RULES_TO_TIER_A = {
    "Duodenum epithelial": STRICT_DUODENUM_MARKERS,
    "Endothelial cells": ["PECAM1", "VWF", "KDR", "CDH5", "PLVAP", "FLT1", "CLDN5", "ESAM", "EMCN", "CD34", "ECSCR", "CLEC14A", "AQP1"],
    "Myeloid cells": ["LST1", "LYZ", "CD68", "AIF1", "CD14", "FCGR3A", "FCGR2A", "CD163", "MPEG1", "CSF1R", "APOE", "CTSD", "PLA2G7", "S100A8", "S100A9", "CXCR1", "CXCR2", "ITGAX", "FGR", "LILRA5", "MCEMP1", "IL1B", "IL1A"],
    "pancreatic acinar epithelium": ["AMY2A", "PRSS1", "PRSS2", "CPA1", "CPA2", "AQP8", "GATM", "KLK11", "REG1A", "REG1B"],
    "Fibroblasts": ["ACTA2", "PDGFRA", "FAP", "THY1", "PDPN", "DCN", "LUM", "COL1A1", "COL1A2", "VIM", "SPARC", "FN1", "VCAN", "FBN1"],
    "T cells": ["CD3D", "CD3E", "CD2", "TRAC", "CD4", "CD8A", "IL7R", "TCF7", "CCL5"],
    "B lineage": ["CD19", "MS4A1", "CD79A", "CD79B", "BANK1", "MZB1", "JCHAIN", "SDC1", "IGHM", "IGKC", "IGHG1", "IGHG2", "IGHG3", "IGHG4"],
    "Mast cells": ["KIT", "CPA3", "MS4A2", "GATA2", "HPGDS", "CTSG"],
    "Islets": ["INS", "GCG", "SST", "PPY", "CHGA", "CHGB", "PCSK2"],
}
TOP_MARKER_RULE_WEIGHTS = {
    "Duodenum epithelial": 2.6,
    "Endothelial cells": 2.5,
    "Myeloid cells": 2.4,
    "T cells": 2.3,
    "B lineage": 2.3,
    "Mast cells": 2.3,
    "Islets": 2.3,
    "pancreatic acinar epithelium": 2.2,
    "Fibroblasts": 2.0,
}
TOP_MARKER_RULE_MIN_HITS = {
    "Duodenum epithelial": 2,
    "Endothelial cells": 2,
    "Myeloid cells": 2,
    "T cells": 2,
    "B lineage": 2,
    "Mast cells": 2,
    "Islets": 2,
    "pancreatic acinar epithelium": 2,
    "Fibroblasts": 2,
}

def palette_for_values(values, base_palette):
    values = pd.Index(pd.Series(values).dropna().astype(str).unique())
    fallback = sns.color_palette("tab20", n_colors=max(len(values), 1)).as_hex()
    out = {}
    for i, value in enumerate(values):
        out[value] = base_palette.get(value, fallback[i % len(fallback)])
    return out

def read_xenium_adata(cfg):
    adata = sc.read_10x_h5(Path(cfg["outs_path"]) / "cell_feature_matrix.h5")
    adata.var_names_make_unique()
    cells = pd.read_csv(Path(cfg["outs_path"]) / "cells.csv.gz").set_index("cell_id")
    obs = adata.obs.join(cells, how="left")
    obs["sample_id"] = cfg["sample_id"]
    obs["display_name"] = cfg["display_name"]
    obs["disease_group"] = cfg["disease_group"]
    obs["panel_hint"] = cfg["panel_hint"]

    adata.obs = obs
    adata = add_xenium_precomputed_clusters(adata, cfg)
    adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].to_numpy(dtype=float)
    adata.layers["counts"] = make_sparse_safe_copy(adata.X)
    return adata

def read_analysis_tar_csv(cfg, member_path):
    archive_path = Path(cfg["outs_path"]) / "analysis.tar.gz"
    if not archive_path.exists():
        return None
    try:
        with tarfile.open(archive_path, "r:gz") as tar:
            handle = tar.extractfile(member_path)
            if handle is None:
                return None
            return pd.read_csv(StringIO(handle.read().decode("utf-8")))
    except KeyError:
        return None

def load_xenium_precomputed_clusters(cfg):
    frames = []
    graphclust = read_analysis_tar_csv(
        cfg,
        "analysis/clustering/gene_expression_graphclust/clusters.csv",
    )
    if graphclust is not None:
        graphclust = graphclust.rename(columns={"Barcode": "cell_id", "Cluster": XENIUM_GRAPHCLUST_KEY})
        graphclust["cell_id"] = graphclust["cell_id"].astype(str)
        graphclust[XENIUM_GRAPHCLUST_KEY] = graphclust[XENIUM_GRAPHCLUST_KEY].astype(str)
        frames.append(graphclust[["cell_id", XENIUM_GRAPHCLUST_KEY]])

    kmeans10 = read_analysis_tar_csv(
        cfg,
        "analysis/clustering/gene_expression_kmeans_10_clusters/clusters.csv",
    )
    if kmeans10 is not None:
        kmeans10 = kmeans10.rename(columns={"Barcode": "cell_id", "Cluster": XENIUM_KMEANS10_KEY})
        kmeans10["cell_id"] = kmeans10["cell_id"].astype(str)
        kmeans10[XENIUM_KMEANS10_KEY] = kmeans10[XENIUM_KMEANS10_KEY].astype(str)
        frames.append(kmeans10[["cell_id", XENIUM_KMEANS10_KEY]])

    groups_path = Path(cfg["outs_path"]) / "cell_groups.csv"
    if groups_path.exists():
        groups = pd.read_csv(groups_path).rename(columns={"group": "xenium_10x_cell_group"})
        groups["cell_id"] = groups["cell_id"].astype(str)
        groups["xenium_10x_cell_group"] = groups["xenium_10x_cell_group"].astype(str)
        frames.append(groups[["cell_id", "xenium_10x_cell_group"]])

    if len(frames) == 0:
        return pd.DataFrame(columns=["cell_id"])

    out = frames[0]
    for frame in frames[1:]:
        out = out.merge(frame, on="cell_id", how="outer")
    return out

def add_xenium_precomputed_clusters(adata, cfg):
    cluster_df = load_xenium_precomputed_clusters(cfg)
    if cluster_df.empty:
        adata.uns["xenium_precomputed_cluster_status"] = "not_found"
        return adata
    cluster_df = cluster_df.drop_duplicates("cell_id").set_index("cell_id")
    for col in cluster_df.columns:
        adata.obs[col] = cluster_df[col].reindex(adata.obs_names).astype("string").fillna("unassigned").astype(str)
    adata.uns["xenium_precomputed_cluster_status"] = "loaded"
    adata.uns["xenium_precomputed_cluster_columns"] = cluster_df.columns.tolist()
    return adata

def choose_annotation_cluster_key(adata):
    for key in ANNOTATION_CLUSTER_PREFERENCE:
        if key in adata.obs.columns and adata.obs[key].astype(str).ne("unassigned").any():
            return key
    if LEIDEN_KEY in adata.obs.columns:
        return LEIDEN_KEY
    raise KeyError("No usable annotation clustering key found.")

def get_matrix(adata, layer=None):
    if layer is None:
        return adata.X
    return adata.layers[layer]

def matrix_mean(adata, genes, layer=None):
    genes = [g for g in genes if g in adata.var_names]
    if len(genes) == 0:
        return np.full(adata.n_obs, np.nan, dtype=float), genes
    X = get_matrix(adata[:, genes], layer=layer)
    if sparse.issparse(X):
        vals = np.asarray(X.mean(axis=1)).ravel()
    else:
        vals = np.asarray(X).mean(axis=1)
    return vals.astype(float), genes

def zscore(values):
    values = np.asarray(values, dtype=float)
    mu = np.nanmean(values)
    sd = np.nanstd(values)
    if not np.isfinite(sd) or np.isclose(sd, 0):
        return np.full(values.shape, np.nan)
    return (values - mu) / sd

def vector_from_gene(adata, gene, layer=None):
    X = get_matrix(adata[:, [gene]], layer=layer)
    vals = X.toarray().ravel() if sparse.issparse(X) else np.asarray(X).ravel()
    return vals.astype(float)

def add_gene_expression_obs(adata, genes, layer="log1p"):
    for gene in genes:
        if gene not in adata.var_names:
            continue
        vals = vector_from_gene(adata, gene, layer=layer)
        adata.obs[f"{gene}_expr"] = vals.astype(float)
        adata.obs[f"{gene}_expr_z"] = zscore(vals)
    return adata

def preprocess_for_independent_annotation(adata):
    sc.pp.filter_cells(adata, min_counts=5)
    sc.pp.filter_genes(adata, min_cells=5)
    adata.layers["counts"] = make_sparse_safe_copy(adata.X)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.layers["log1p"] = make_sparse_safe_copy(adata.X)

    # Use the full retained Xenium panel for PCA, not a cross-sample HVG subset.
    n_comps = int(min(40, max(2, adata.n_vars - 1), max(2, adata.n_obs - 1)))
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, n_comps=n_comps, svd_solver="arpack", random_state=42)
    sc.pp.neighbors(adata, n_neighbors=20, n_pcs=min(30, n_comps), random_state=42)
    sc.tl.umap(adata, min_dist=0.35, random_state=42)
    sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION, key_added=LEIDEN_KEY, random_state=42)
    return adata

def add_marker_scores(adata):
    resolved_sets = {}
    for set_name, genes in GENE_SETS.items():
        vals, present = matrix_mean(adata, genes, layer="log1p")
        adata.obs[f"{set_name}_score"] = vals
        adata.obs[f"{set_name}_score_z"] = zscore(vals)
        resolved_sets[set_name] = present
    adata = add_gene_expression_obs(adata, MARKER_EXPORT_GENES, layer="log1p")
    adata.uns["xenium_marker_sets_resolved"] = resolved_sets
    return adata

def top_markers_by_cluster(adata, cluster_key=LEIDEN_KEY, n_top=8):
    X = adata.layers["log1p"]
    if sparse.issparse(X):
        global_mean = np.asarray(X.mean(axis=0)).ravel()
    else:
        global_mean = np.asarray(X).mean(axis=0)

    out = {}
    clusters = adata.obs[cluster_key].astype(str)
    for cluster in sorted(clusters.unique(), key=lambda x: (len(x), x)):
        mask = clusters.to_numpy() == cluster
        if sparse.issparse(X):
            mean = np.asarray(X[mask].mean(axis=0)).ravel()
        else:
            mean = np.asarray(X[mask]).mean(axis=0)
        diff = mean - global_mean
        order = np.argsort(diff)[::-1][:n_top]
        out[cluster] = ", ".join(adata.var_names[order].tolist())
    return out

def summarize_annotation_clusters(adata, cfg, cluster_key=None):
    cluster_key = cluster_key or choose_annotation_cluster_key(adata)
    clusters = adata.obs[cluster_key].astype(str)
    top_markers = top_markers_by_cluster(adata, cluster_key=cluster_key)
    score_cols = [f"{name}_score_z" for name in GENE_SETS if f"{name}_score_z" in adata.obs.columns]
    marker_genes = [g for g in VALIDATION_MARKERS if g in adata.var_names and f"{g}_expr" in adata.obs.columns]

    rows = []
    for cluster in sorted(clusters.unique(), key=lambda x: (len(x), x)):
        mask = clusters == cluster
        row = {
            "sample_id": cfg["sample_id"],
            "display_name": cfg["display_name"],
            "disease_group": cfg["disease_group"],
            "cluster_key": cluster_key,
            "leiden": cluster,
            "n_cells": int(mask.sum()),
            "fraction": float(mask.mean()),
            "top_marker_genes": top_markers.get(cluster, ""),
        }
        for col in score_cols:
            row[col] = float(pd.to_numeric(adata.obs.loc[mask, col], errors="coerce").mean())
        for gene in marker_genes:
            expr = pd.to_numeric(adata.obs.loc[mask, f"{gene}_expr"], errors="coerce")
            row[f"{gene}_mean"] = float(expr.mean())
            row[f"{gene}_frac_pos"] = float((expr > 0).mean())
        rows.append(row)
    return pd.DataFrame(rows)

def suggest_tier_a_from_cluster(row):
    candidates = []
    for score_col, label in BROAD_SCORE_TO_TIER_A.items():
        z_col = score_col.replace("_score", "_score_z")
        if z_col in row and pd.notna(row[z_col]):
            candidates.append((float(row[z_col]), label, z_col))

    # Unsupervised marker genes are treated as additional evidence. This is
    # especially important for panels where canonical markers are missing from
    # a curated score, e.g. pdac_io_v1 endothelial cells with PLVAP/FLT1.
    for label, markers in TOP_MARKER_RULES_TO_TIER_A.items():
        marker_hits = top_marker_count(row, markers)
        if marker_hits >= TOP_MARKER_RULE_MIN_HITS.get(label, 1):
            score_cols = [
                score_col.replace("_score", "_score_z")
                for score_col, score_label in BROAD_SCORE_TO_TIER_A.items()
                if score_label == label
            ]
            score_vals = [row_value(row, col, np.nan) for col in score_cols]
            score_vals = [v for v in score_vals if pd.notna(v)]
            base_score = max(score_vals) if len(score_vals) > 0 else 0.0
            rule_score = TOP_MARKER_RULE_WEIGHTS.get(label, 1.75)
            candidates.append((max(float(base_score), rule_score), label, f"top_marker_rule_{marker_hits}_hits"))

    if not candidates:
        return "Unknown", np.nan, "no marker program available"
    candidates = sorted(candidates, reverse=True)
    supported = []
    unsupported = []
    for score, label, col in candidates:
        if label_supported_by_anchors(row, label):
            supported.append((score, label, col))
        else:
            unsupported.append((score, label, col))

    if len(supported) == 0:
        return "Unknown", candidates[0][0], f"no candidate label passed anchor checks; best was {candidates[0][2]}"

    best_score, best_label, best_col = supported[0]
    second_score = supported[1][0] if len(supported) > 1 else np.nan
    if not np.isfinite(best_score) or best_score < -0.15:
        return "Unknown", best_score, f"weak best score: {best_col}"
    if np.isfinite(second_score) and (best_score - second_score) < 0.08:
        note = f"ambiguous: {best_col} close to second-best"
    else:
        note = f"best marker program: {best_col}"
    if unsupported and unsupported[0][0] > best_score:
        note += f"; vetoed unsupported {unsupported[0][2]}"
    return best_label, best_score, note

def row_value(row, key, default=np.nan):
    return row[key] if key in row and pd.notna(row[key]) else default

def top_marker_hit(row, markers):
    return top_marker_count(row, markers) > 0

def top_marker_count(row, markers):
    top = str(row_value(row, "top_marker_genes", "")).replace(" ", "")
    top_markers = set(top.split(",")) if top else set()
    return sum(marker in top_markers for marker in markers)

def max_marker_mean(row, markers):
    vals = [row_value(row, f"{marker}_mean", np.nan) for marker in markers]
    vals = [v for v in vals if pd.notna(v)]
    return np.nanmax(vals) if vals else np.nan

def max_marker_frac(row, markers):
    vals = [row_value(row, f"{marker}_frac_pos", np.nan) for marker in markers]
    vals = [v for v in vals if pd.notna(v)]
    return np.nanmax(vals) if vals else np.nan

def label_supported_by_anchors(row, label):
    # Ductal and endocrine classes can use marker programs directly. Acinar and
    # duodenum calls are checked with top-marker anchors because some panels have
    # only weak overlapping epithelial genes.
    if label in {"pancreatic ductal epithelium", "Islets"}:
        return True

    anchor_markers = {
        "Duodenum epithelial": STRICT_DUODENUM_MARKERS,
        "pancreatic acinar epithelium": ["AMY2A", "PRSS1", "PRSS2", "CPA1", "CPA2", "AQP8", "GATM", "KLK11", "REG1A", "REG1B"],
        "Fibroblasts": ["PDGFRA", "ACTA2", "THY1", "PDPN", "DCN", "LUM", "COL1A1", "VIM", "SPARC", "FN1", "VCAN", "FBN1"],
        "Endothelial cells": ["PECAM1", "VWF", "KDR", "CDH5", "SOX17", "ACKR1", "ADGRL4", "PLVAP", "FLT1", "CLDN5", "ESAM", "EMCN", "CD34", "ECSCR", "CLEC14A", "AQP1"],
        "T cells": ["CD3D", "CD3E", "CD2", "TRAC", "CD4", "CD8A", "IL7R", "TCF7", "CCL5"],
        "B lineage": ["CD19", "MS4A1", "CD79A", "CD79B", "BANK1", "MZB1", "JCHAIN", "SDC1", "TNFRSF17", "IGHM", "IGKC", "IGHG1", "IGHG2", "IGHG3", "IGHG4", "IGHA1", "IGHA2"],
        "Myeloid cells": ["LST1", "LYZ", "CD14", "CD68", "AIF1", "FCGR3A", "FCGR1A", "FCGR2A", "C1QA", "C1QB", "C1QC", "CD163", "MPEG1", "CSF1R", "APOE", "CTSD", "PLA2G7", "S100A8", "S100A9", "CXCR1", "CXCR2", "ITGAX", "FGR", "LILRA5", "MCEMP1", "IL1B", "IL1A"],
        "Mast cells": ["KIT", "CPA3", "MS4A2", "GATA2", "HPGDS", "CTSG"],
    }.get(label, [])

    if len(anchor_markers) == 0:
        return True
    max_mean = max_marker_mean(row, anchor_markers)
    max_frac = max_marker_frac(row, anchor_markers)
    ptprc_mean = row_value(row, "PTPRC_mean", 0)

    if label in {"T cells", "B lineage", "Myeloid cells"}:
        if top_marker_count(row, anchor_markers) >= 3:
            return True
        if top_marker_count(row, anchor_markers) >= 2 and ptprc_mean >= 1.2:
            return True
        return (ptprc_mean >= 2.0) and (
            (pd.notna(max_mean) and max_mean >= 1.2)
            or (pd.notna(max_frac) and max_frac >= 0.25)
        )
    if label == "Endothelial cells":
        return (
            (pd.notna(max_mean) and max_mean >= 1.0)
            or (pd.notna(max_frac) and max_frac >= 0.18)
            or top_marker_hit(row, anchor_markers)
        )
    if label == "Duodenum epithelial":
        # REG4/DMBT1/TFF3/TMPRSS2 are deliberately excluded here because they
        # can be PDAC/PanIN-like ductal remodeling. A duodenum call needs either
        # two strict intestinal markers, or CDX2 plus another strict intestinal
        # marker such as KRT20/VIL1/FABP/ALPI/MUC2.
        strict_hits = top_marker_count(row, STRICT_DUODENUM_MARKERS)
        cdx2_frac = row_value(row, "CDX2_frac_pos", np.nan)
        cdx2_mean = row_value(row, "CDX2_mean", np.nan)
        non_cdx2_frac = max_marker_frac(row, NON_CDX2_DUODENUM_MARKERS)
        non_cdx2_mean = max_marker_mean(row, NON_CDX2_DUODENUM_MARKERS)
        has_cdx2 = (
            top_marker_hit(row, ["CDX2"])
            or (pd.notna(cdx2_frac) and cdx2_frac >= 0.25)
            or (pd.notna(cdx2_mean) and cdx2_mean >= 1.0)
        )
        has_second_intestinal_anchor = (
            top_marker_hit(row, NON_CDX2_DUODENUM_MARKERS)
            or (pd.notna(non_cdx2_frac) and non_cdx2_frac >= 0.20)
            or (pd.notna(non_cdx2_mean) and non_cdx2_mean >= 1.0)
        )
        return strict_hits >= 2 or (has_cdx2 and has_second_intestinal_anchor)
    if top_marker_hit(row, anchor_markers):
        return True
    if label == "Mast cells":
        return (
            pd.notna(max_mean)
            and pd.notna(max_frac)
            and max_mean >= 1.5
            and max_frac >= 0.30
        )
    return (
        (pd.notna(max_mean) and max_mean >= 1.0)
        or (pd.notna(max_frac) and max_frac >= 0.20)
    )

def suggest_tier_b_from_cluster(row, tier_a):
    if tier_a == "pancreatic ductal epithelium":
        intestinal_like_score = row_value(row, "intestinal_like_ductal_remodeling_score_z", np.nan)
        intestinal_like_hits = top_marker_count(row, INTESTINAL_LIKE_DUCTAL_MARKERS)
        cdx2_frac = row_value(row, "CDX2_frac_pos", np.nan)
        support_frac = max_marker_frac(row, ["REG4", "DMBT1", "TMPRSS2", "GPX2", "TFF3"])
        panin = np.nanmax([
            row_value(row, "panin_mucin_remodeling_score_z"),
            row_value(row, "MUC5AC_mean"),
            row_value(row, "TFF1_mean"),
            row_value(row, "TFF2_mean"),
            row_value(row, "TFF3_mean"),
            row_value(row, "CEACAM6_mean"),
        ])
        prolif = np.nanmax([row_value(row, "proliferation_score_z"), row_value(row, "MKI67_mean"), row_value(row, "TOP2A_mean")])
        if (
            intestinal_like_hits >= 2
            or (
                pd.notna(cdx2_frac)
                and cdx2_frac >= 0.15
                and (
                    (pd.notna(support_frac) and support_frac >= 0.25)
                    or (np.isfinite(intestinal_like_score) and intestinal_like_score > 0.75)
                )
            )
            or (
                np.isfinite(intestinal_like_score)
                and intestinal_like_score > 1.0
                and top_marker_hit(row, INTESTINAL_LIKE_DUCTAL_MARKERS)
            )
        ):
            return "intestinal-like/PanIN-like ductal epithelial"
        if np.isfinite(prolif) and prolif > 0.35:
            return "proliferative ductal/tumor epithelial"
        if np.isfinite(panin) and panin > 0.35:
            return "mucin/PanIN-like ductal epithelial"
        return "ductal/tumor epithelial"
    if tier_a == "Duodenum epithelial":
        return "duodenum/intestinal epithelial"
    if tier_a == "pancreatic acinar epithelium":
        return "acinar epithelial"
    if tier_a == "Islets":
        return "endocrine/islet"
    if tier_a == "Fibroblasts":
        if row_value(row, "ACTA2_mean", 0) > 0 or row_value(row, "fibroblast_stellate_score_z", 0) > 0.6:
            return "activated fibroblast/stellate"
        return "fibroblast/stellate"
    if tier_a == "Endothelial cells":
        if top_marker_hit(row, ["PLVAP", "FLT1", "RGS5", "SPARCL1", "IGFBP7"]):
            return "endothelial/perivascular"
        return "endothelial"
    if tier_a == "T cells":
        if row_value(row, "FOXP3_mean", 0) > 0:
            return "Tregs"
        if row_value(row, "CD8A_mean", 0) > row_value(row, "CD4_mean", 0):
            return "CD8 T cells"
        if row_value(row, "cytotoxic_t_nk_score_z", 0) > 0.4:
            return "cytotoxic T/NK-like"
        return "CD4/other T cells"
    if tier_a == "B lineage":
        if row_value(row, "plasma_cell_score_z", 0) > row_value(row, "b_cell_score_z", 0):
            return "plasma-like B lineage"
        return "B cells"
    if tier_a == "Myeloid cells":
        if top_marker_hit(row, ["S100A8", "S100A9", "CXCR1", "CXCR2", "IL1B", "IL1A", "MCEMP1"]):
            return "inflammatory myeloid"
        return "myeloid/macrophage"
    if tier_a == "Mast cells":
        return "mast cell"
    return "Unknown"

def add_suggested_cluster_labels(cluster_df):
    cluster_df = cluster_df.copy()
    suggested = cluster_df.apply(lambda row: suggest_tier_a_from_cluster(row), axis=1)
    cluster_df["suggested_Tier_A"] = [x[0] for x in suggested]
    cluster_df["annotation_score"] = [x[1] for x in suggested]
    cluster_df["annotation_note"] = [x[2] for x in suggested]
    cluster_df["suggested_Tier_B"] = [
        suggest_tier_b_from_cluster(row, row["suggested_Tier_A"])
        for _, row in cluster_df.iterrows()
    ]
    return cluster_df

def apply_curated_cluster_label_overrides(review_df):
    review_df = review_df.copy()
    for (sample_id, cluster_key, cluster), override in CURATED_CLUSTER_LABEL_OVERRIDES.items():
        mask = (
            review_df["sample_id"].astype(str).eq(sample_id)
            & review_df["cluster_key"].astype(str).eq(cluster_key)
            & review_df["leiden"].astype(str).eq(str(cluster))
        )
        if not mask.any():
            continue
        for col, value in override.items():
            review_df.loc[mask, col] = value
        review_df.loc[mask, "annotation_note"] = (
            review_df.loc[mask, "annotation_note"].astype(str)
            + "; curated graphclust override"
        )
    return review_df

def update_cluster_review_table(cluster_df):
    review_cols = [
        "annotation_version", "sample_id", "cluster_key", "leiden", "n_cells", "fraction", "top_marker_genes",
        "suggested_Tier_A", "suggested_Tier_B", "annotation_score", "annotation_note",
        "final_Tier_A", "final_Tier_B", "notes",
    ]
    new_review = cluster_df.copy()
    new_review["annotation_version"] = ANNOTATION_VERSION
    new_review["final_Tier_A"] = new_review["suggested_Tier_A"]
    new_review["final_Tier_B"] = new_review["suggested_Tier_B"]
    new_review["notes"] = ""
    new_review = new_review[[c for c in review_cols if c in new_review.columns]]

    if CLUSTER_REVIEW_PATH.exists():
        old = pd.read_csv(CLUSTER_REVIEW_PATH, dtype={"leiden": str})
        if "annotation_version" in old.columns:
            old = old.loc[old["annotation_version"] == ANNOTATION_VERSION].copy()
        else:
            old = old.iloc[0:0].copy()
        if "cluster_key" not in old.columns:
            old["cluster_key"] = ""
        keep_cols = ["sample_id", "cluster_key", "leiden", "final_Tier_A", "final_Tier_B", "notes"]
        old_keep = old.reindex(columns=keep_cols).dropna(subset=["sample_id", "leiden"], how="any").copy()
        if "cluster_key" not in old_keep.columns or old_keep["cluster_key"].isna().all():
            old_keep["cluster_key"] = ""
            new_review["cluster_key"] = new_review.get("cluster_key", "")
        merge_cols = ["sample_id", "cluster_key", "leiden"]
        new_review = new_review.drop(columns=["final_Tier_A", "final_Tier_B", "notes"], errors="ignore").merge(
            old_keep,
            on=merge_cols,
            how="left",
        )
        new_review["final_Tier_A"] = new_review["final_Tier_A"].fillna(new_review["suggested_Tier_A"])
        new_review["final_Tier_B"] = new_review["final_Tier_B"].fillna(new_review["suggested_Tier_B"])
        new_review["notes"] = new_review["notes"].fillna("")

        old_other = old.loc[~old.set_index(merge_cols).index.isin(
            new_review.set_index(merge_cols).index
        )]
        if len(old_other) > 0:
            new_review = pd.concat([old_other.reindex(columns=new_review.columns), new_review], ignore_index=True)

    new_review = apply_curated_cluster_label_overrides(new_review)
    new_review = new_review.sort_values(["sample_id", "cluster_key", "leiden"], key=lambda s: s.astype(str))
    new_review.to_csv(CLUSTER_REVIEW_PATH, index=False)
    return new_review

def apply_review_labels(adata, review_df, cfg):
    sample_review = review_df.loc[review_df["sample_id"] == cfg["sample_id"]].copy()
    sample_review["leiden"] = sample_review["leiden"].astype(str)
    if "cluster_key" in sample_review.columns and sample_review["cluster_key"].notna().any():
        cluster_key = sample_review["cluster_key"].dropna().astype(str).mode().iat[0]
    else:
        cluster_key = choose_annotation_cluster_key(adata)
    tier_a_map = sample_review.set_index("leiden")["final_Tier_A"].to_dict()
    tier_b_map = sample_review.set_index("leiden")["final_Tier_B"].to_dict()
    adata.obs["spatioev_annotation_cluster_key"] = cluster_key
    adata.obs["spatioev_annotation_cluster"] = adata.obs[cluster_key].astype(str)
    adata.uns["spatioev_annotation_cluster_key"] = cluster_key
    adata.obs["Tier_A"] = adata.obs[cluster_key].astype(str).map(tier_a_map).fillna("Unknown").astype(str)
    adata.obs["Tier_B"] = adata.obs[cluster_key].astype(str).map(tier_b_map).fillna("Unknown").astype(str)
    adata.obs["Tier_A_auto"] = adata.obs["Tier_A"]
    adata.obs["Tier_B_auto"] = adata.obs["Tier_B"]
    return adata

def refine_epithelial_labels_by_subclustering(adata, cfg):
    """Refine epithelial labels after broad all-cell Leiden annotation.

    The first pass assigns one label per whole-sample Leiden cluster. That can
    hide normal ductal cells inside large exocrine/acinar clusters. Here we run a
    focused epithelial-only PCA + MiniBatchKMeans pass across ductal, acinar,
    islet, and duodenum candidates, then interpret each subcluster using the
    marker-program z-scores already computed per cell. Non-epithelial labels are
    deliberately left untouched.
    """
    from sklearn.cluster import MiniBatchKMeans
    from sklearn.decomposition import PCA

    adata.obs["Tier_A_cluster_level"] = adata.obs["Tier_A"].astype(str)
    adata.obs["Tier_B_cluster_level"] = adata.obs["Tier_B"].astype(str)
    adata.obs["epithelial_refinement_cluster"] = "not_run"
    adata.obs["epithelial_refinement_label"] = "not_run"

    available_genes = set(adata.var_names)
    if not set(EPITHELIAL_REFINEMENT_REQUIRED_GENES).issubset(available_genes):
        adata.uns["epithelial_refinement_status"] = "skipped_required_genes_missing"
        return adata, pd.DataFrame()
    if "log1p" not in adata.layers.keys():
        adata.uns["epithelial_refinement_status"] = "skipped_log1p_layer_missing"
        return adata, pd.DataFrame()

    candidate_mask = adata.obs["Tier_A"].astype(str).isin(EPITHELIAL_REFINEMENT_CANDIDATE_TIER_A).to_numpy()
    if int(candidate_mask.sum()) < EPITHELIAL_REFINEMENT_MIN_CELLS:
        adata.uns["epithelial_refinement_status"] = "skipped_too_few_epithelial_candidates"
        return adata, pd.DataFrame()

    X = adata.layers["log1p"][candidate_mask, :]
    if sparse.issparse(X):
        X = X.toarray()
    X = np.asarray(X, dtype=np.float32)
    gene_keep = (X > 0).sum(axis=0) >= 20
    if int(gene_keep.sum()) < 5:
        adata.uns["epithelial_refinement_status"] = "skipped_too_few_expressed_genes"
        return adata, pd.DataFrame()

    X = X[:, gene_keep]
    genes = np.asarray(adata.var_names)[gene_keep]
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std[std == 0] = 1
    Xz = (X - mean) / std

    n_components = int(min(30, Xz.shape[1] - 1, Xz.shape[0] - 1))
    if n_components < 2:
        adata.uns["epithelial_refinement_status"] = "skipped_too_few_components"
        return adata, pd.DataFrame()

    pca = PCA(n_components=n_components, svd_solver="randomized", random_state=42)
    embedding = pca.fit_transform(Xz)
    n_clusters = int(min(EPITHELIAL_REFINEMENT_K, max(4, candidate_mask.sum() // 2500)))
    kmeans = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=42,
        batch_size=4096,
        n_init=10,
    )
    subcluster_labels = kmeans.fit_predict(embedding).astype(str)

    candidate_obs_names = adata.obs_names[candidate_mask]
    candidate_obs = adata.obs.loc[candidate_obs_names].copy()
    annotation_cluster_key = choose_annotation_cluster_key(adata)
    adata.obs.loc[candidate_obs_names, "epithelial_refinement_cluster"] = subcluster_labels

    marker_genes = [
        "EPCAM", "KRT7", "KRT8", "KRT18", "KRT19", "SOX9", "MUC1",
        "CFTR", "FXYD2", "TM4SF4", "PROX1", "EHF", "MUC5AC", "TFF1",
        "TFF2", "TFF3", "CEACAM6", "AGR2", "AGR3", "CDX2", "REG4",
        "DMBT1", "TMPRSS2", "GPX2", "KRT20", "MUC2", "VIL1", "FABP1",
        "FABP2", "ALPI", "AMY2A", "PRSS1", "PRSS2", "CPA1", "CPA2",
        "AQP8", "GATM", "ANPEP", "KLK11", "INS", "GCG", "SST", "PPY",
        "CHGA", "MKI67", "UBE2C", "TOP2A",
    ]
    marker_genes = [gene for gene in marker_genes if gene in genes]
    gene_to_idx = {gene: idx for idx, gene in enumerate(genes)}
    marker_expr = pd.DataFrame(
        X[:, [gene_to_idx[gene] for gene in marker_genes]],
        index=candidate_obs_names,
        columns=marker_genes,
    )

    present_counts = {
        set_name: len([gene for gene in geneset if gene in available_genes])
        for set_name, geneset in GENE_SETS.items()
    }
    acinar_ready = present_counts.get("acinar_epithelial", 0) >= 3
    islet_ready = present_counts.get("islet_endocrine", 0) >= 2
    duodenum_ready = all(gene in available_genes for gene in ["CDX2", "REG4", "DMBT1", "TMPRSS2"])

    score_cols = [
        "ductal_epithelial_score_z",
        "duodenum_epithelial_score_z",
        "acinar_epithelial_score_z",
        "islet_endocrine_score_z",
        "panin_mucin_remodeling_score_z",
        "intestinal_like_ductal_remodeling_score_z",
        "proliferation_score_z",
    ]
    score_cols = [col for col in score_cols if col in candidate_obs.columns]

    def finite_mean(values):
        values = [value for value in values if pd.notna(value)]
        return float(np.mean(values)) if len(values) > 0 else np.nan

    def classify_epithelial_subcluster(row):
        parent_a = row.get("parent_Tier_A_major", "Unknown")
        parent_b = row.get("parent_Tier_B_major", "Unknown")
        ductal = row_value(row, "ductal_epithelial_score_z", np.nan)
        acinar = row_value(row, "acinar_epithelial_score_z", np.nan)
        islet = row_value(row, "islet_endocrine_score_z", np.nan)
        duodenum = row_value(row, "duodenum_epithelial_score_z", np.nan)

        is_duodenum_like = (
            duodenum_ready
            and row.get("CDX2_frac_pos", 0) >= 0.25
            and row.get("REG4_frac_pos", 0) >= 0.60
            and row.get("DMBT1_frac_pos", 0) >= 0.65
            and row.get("TMPRSS2_frac_pos", 0) >= 0.25
            and row["duodenum_minus_ductal_score"] >= 1.50
        )
        if is_duodenum_like:
            return "Duodenum epithelial", "duodenum/intestinal epithelial"

        # Preserve existing duodenum calls unless the subcluster is clearly a
        # ductal/PanIN-like population. This avoids undoing the pdac_io_v1
        # duodenum refinement unless the marker programs strongly disagree.
        if parent_a == "Duodenum epithelial":
            if np.isfinite(ductal) and np.isfinite(duodenum) and ductal > duodenum + 0.75 and ductal > 0.75:
                return "pancreatic ductal epithelium", suggest_tier_b_from_cluster(pd.Series(row), "pancreatic ductal epithelium")
            return parent_a, parent_b

        # Existing ductal calls are trusted unless a subcluster has very strong
        # endocrine support. The goal here is mainly to rescue missed ductal
        # cells, not aggressively remove ductal cells from PDAC regions.
        if parent_a == "pancreatic ductal epithelium":
            if islet_ready and np.isfinite(islet) and islet > max(ductal, acinar) + 1.00 and islet > 1.00:
                return "Islets", "endocrine/islet"
            return parent_a, parent_b

        if parent_a == "Islets":
            if np.isfinite(ductal) and ductal > islet + 1.00 and ductal > 1.00:
                return "pancreatic ductal epithelium", suggest_tier_b_from_cluster(pd.Series(row), "pancreatic ductal epithelium")
            return parent_a, parent_b

        if (
            parent_a == "pancreatic acinar epithelium"
            and np.isfinite(ductal)
            and ductal > 0
            and (
                (np.isfinite(acinar) and ductal > acinar + 0.25)
                or not acinar_ready
            )
        ):
            return "pancreatic ductal epithelium", suggest_tier_b_from_cluster(pd.Series(row), "pancreatic ductal epithelium")

        if (
            parent_a == "pancreatic acinar epithelium"
            and acinar_ready
            and np.isfinite(acinar)
            and acinar > 0
        ):
            return "pancreatic acinar epithelium", "acinar epithelial"

        return parent_a, parent_b

    rows = []
    refined_tier_a_by_subcluster = {}
    refined_tier_b_by_subcluster = {}
    for subcluster in sorted(pd.unique(subcluster_labels), key=lambda x: (len(str(x)), str(x))):
        idx = subcluster_labels == subcluster
        obs_idx = candidate_obs_names[idx]
        row = {
            "sample_id": cfg["sample_id"],
            "method": f"epithelial_pca_kmeans_{n_clusters}",
            "subcluster": str(subcluster),
            "n_cells": int(idx.sum()),
            "fraction_of_epithelial_candidates": float(idx.mean()),
        }
        parent_leiden_mix = candidate_obs.loc[obs_idx, annotation_cluster_key].astype(str).value_counts(normalize=True)
        parent_tier_a_mix = candidate_obs.loc[obs_idx, "Tier_A"].astype(str).value_counts(normalize=True)
        parent_tier_b_mix = candidate_obs.loc[obs_idx, "Tier_B"].astype(str).value_counts(normalize=True)
        row["parent_annotation_cluster_key"] = annotation_cluster_key
        row["parent_annotation_cluster_mix"] = "; ".join(f"{label}:{frac:.2f}" for label, frac in parent_leiden_mix.head(4).items())
        row["parent_leiden_mix"] = row["parent_annotation_cluster_mix"]
        row["parent_Tier_A_mix"] = "; ".join(f"{label}:{frac:.2f}" for label, frac in parent_tier_a_mix.head(4).items())
        row["parent_Tier_B_mix"] = "; ".join(f"{label}:{frac:.2f}" for label, frac in parent_tier_b_mix.head(4).items())
        row["parent_Tier_A_major"] = parent_tier_a_mix.index[0]
        row["parent_Tier_B_major"] = parent_tier_b_mix.index[0]

        for col in score_cols:
            row[col] = float(pd.to_numeric(candidate_obs.loc[obs_idx, col], errors="coerce").mean())
        for gene in marker_genes:
            vals = marker_expr.loc[obs_idx, gene]
            row[f"{gene}_mean"] = float(vals.mean())
            row[f"{gene}_frac_pos"] = float((vals > 0).mean())

        duodenum_panel_score = finite_mean([
            row.get("CDX2_mean", np.nan),
            row.get("REG4_mean", np.nan),
            row.get("DMBT1_mean", np.nan),
            row.get("TMPRSS2_mean", np.nan),
        ])
        ductal_tumor_score = finite_mean([
            row.get("SOX9_mean", np.nan),
            row.get("CEACAM6_mean", np.nan),
            row.get("MUC5AC_mean", np.nan),
            row.get("TFF3_mean", np.nan),
        ])
        row["duodenum_panel_score"] = float(duodenum_panel_score)
        row["ductal_tumor_score"] = float(ductal_tumor_score)
        row["duodenum_minus_ductal_score"] = float(duodenum_panel_score - ductal_tumor_score)

        refined_tier_a, refined_tier_b = classify_epithelial_subcluster(row)
        row["refined_Tier_A"] = refined_tier_a
        row["refined_Tier_B"] = refined_tier_b
        row["refinement_action"] = (
            "kept"
            if refined_tier_a == row["parent_Tier_A_major"] and refined_tier_b == row["parent_Tier_B_major"]
            else f"{row['parent_Tier_A_major']} -> {refined_tier_a}"
        )
        refined_tier_a_by_subcluster[str(subcluster)] = refined_tier_a
        refined_tier_b_by_subcluster[str(subcluster)] = refined_tier_b
        rows.append(row)

    refinement_df = pd.DataFrame(rows)
    refined_tier_a = pd.Series(subcluster_labels).map(refined_tier_a_by_subcluster).to_numpy()
    refined_tier_b = pd.Series(subcluster_labels).map(refined_tier_b_by_subcluster).to_numpy()

    adata.obs.loc[candidate_obs_names, "Tier_A"] = refined_tier_a
    adata.obs.loc[candidate_obs_names, "Tier_B"] = refined_tier_b
    adata.obs.loc[candidate_obs_names, "epithelial_refinement_label"] = refined_tier_a

    n_changed = int((candidate_obs["Tier_A"].astype(str).to_numpy() != refined_tier_a).sum())
    n_rescued_ductal = int(
        (
            (candidate_obs["Tier_A"].astype(str).to_numpy() != "pancreatic ductal epithelium")
            & (refined_tier_a == "pancreatic ductal epithelium")
        ).sum()
    )
    adata.uns["epithelial_refinement_status"] = (
        f"completed_{n_clusters}_subclusters_{n_changed}_changed_{n_rescued_ductal}_ductal_rescued"
    )
    refinement_path = ANNOTATION_QC_DIR / f"{cfg['sample_id']}_epithelial_refinement_qc.csv"
    refinement_df.to_csv(refinement_path, index=False)

    assignment_df = pd.DataFrame(
        {
            "cell_id": candidate_obs_names,
            "epithelial_refinement_cluster": subcluster_labels,
            "original_Tier_A": candidate_obs["Tier_A"].astype(str).to_numpy(),
            "original_Tier_B": candidate_obs["Tier_B"].astype(str).to_numpy(),
            "refined_Tier_A": refined_tier_a,
            "refined_Tier_B": refined_tier_b,
        }
    )
    assignment_df.to_csv(ANNOTATION_QC_DIR / f"{cfg['sample_id']}_epithelial_refinement_assignments.csv", index=False)
    return adata, refinement_df


In [ ]:

FORCE_REANNOTATE = False
summary_frames = []
cluster_summary_frames = []
marker_availability_rows = []

for cfg in SAMPLE_CONFIGS:
    out_path = ANNOTATED_DIR / f"{cfg['sample_id']}_annotated.h5ad"
    if out_path.exists() and not FORCE_REANNOTATE:
        adata = sc.read_h5ad(out_path, backed="r")
        cache_ok = (
            LEIDEN_KEY in adata.obs.columns
            and "Tier_A" in adata.obs.columns
            and (
                XENIUM_GRAPHCLUST_KEY in adata.obs.columns
                or adata.uns.get("xenium_precomputed_cluster_status") == "not_found"
            )
            and adata.uns.get("spatioev_xenium_annotation_version") == ANNOTATION_VERSION
        )
        can_fast_relabel = (
            LEIDEN_KEY in adata.obs.columns
            and "log1p" in adata.layers.keys()
        )
        if not cache_ok:
            adata.file.close()
            if can_fast_relabel:
                print(f"Cached annotation is older, but Leiden/log1p are available; fast relabeling {cfg['sample_id']}.")
                adata = sc.read_h5ad(out_path)
                adata = add_xenium_precomputed_clusters(adata, cfg)
                available_genes = set(adata.var_names)
                for set_name, genes in GENE_SETS.items():
                    present = [g for g in genes if g in available_genes]
                    marker_availability_rows.append(
                        {
                            "sample_id": cfg["sample_id"],
                            "gene_set": set_name,
                            "n_requested": len(genes),
                            "n_present": len(present),
                            "present_genes": ", ".join(present),
                        }
                    )
                adata = add_marker_scores(adata)
                cluster_summary_df = summarize_annotation_clusters(adata, cfg)
                cluster_summary_df = add_suggested_cluster_labels(cluster_summary_df)
                review_df = update_cluster_review_table(cluster_summary_df)
                adata = apply_review_labels(adata, review_df, cfg)
                adata, refinement_df = refine_epithelial_labels_by_subclustering(adata, cfg)
                adata.uns["spatioev_xenium_annotation_version"] = ANNOTATION_VERSION
                cluster_summary_df.to_csv(ANNOTATION_QC_DIR / f"{cfg['sample_id']}_cluster_annotation_summary.csv", index=False)
                adata.write_h5ad(out_path)
                cluster_summary_frames.append(cluster_summary_df)

                counts = adata.obs["Tier_A"].value_counts().rename("n").reset_index()
                counts.columns = ["Tier_A", "n"]
                counts["sample_id"] = cfg["sample_id"]
                counts["disease_group"] = cfg["disease_group"]
                summary_frames.append(counts)
                del adata
                gc.collect()
                continue
            print(f"Cached annotation is from an older workflow and cannot be fast-relabelled; rebuilding {cfg['sample_id']}.")
        else:
            print(f"Using cached annotation: {out_path}")
            counts = adata.obs["Tier_A"].value_counts().rename("n").reset_index()
            counts.columns = ["Tier_A", "n"]
            counts["sample_id"] = cfg["sample_id"]
            counts["disease_group"] = cfg["disease_group"]
            summary_frames.append(counts)
            cluster_path = ANNOTATION_QC_DIR / f"{cfg['sample_id']}_cluster_annotation_summary.csv"
            if cluster_path.exists():
                cluster_summary_frames.append(pd.read_csv(cluster_path, dtype={"leiden": str}))
            adata.file.close()
            continue

    if out_path.exists() and FORCE_REANNOTATE:
        print(f"FORCE_REANNOTATE=True; rebuilding {cfg['sample_id']}.")
    elif not out_path.exists():
        print(f"No cached annotation found for {cfg['sample_id']}.")

    print(f"Clustering and annotating {cfg['sample_id']} with its own full panel...")
    adata = read_xenium_adata(cfg)
    available_genes = set(adata.var_names)
    for set_name, genes in GENE_SETS.items():
        present = [g for g in genes if g in available_genes]
        marker_availability_rows.append(
            {
                "sample_id": cfg["sample_id"],
                "gene_set": set_name,
                "n_requested": len(genes),
                "n_present": len(present),
                "present_genes": ", ".join(present),
            }
        )

    adata = preprocess_for_independent_annotation(adata)
    adata = add_marker_scores(adata)
    cluster_summary_df = summarize_annotation_clusters(adata, cfg)
    cluster_summary_df = add_suggested_cluster_labels(cluster_summary_df)
    review_df = update_cluster_review_table(cluster_summary_df)
    adata = apply_review_labels(adata, review_df, cfg)
    adata, refinement_df = refine_epithelial_labels_by_subclustering(adata, cfg)
    adata.uns["spatioev_xenium_annotation_version"] = ANNOTATION_VERSION

    cluster_summary_df.to_csv(ANNOTATION_QC_DIR / f"{cfg['sample_id']}_cluster_annotation_summary.csv", index=False)
    adata.write_h5ad(out_path)
    cluster_summary_frames.append(cluster_summary_df)

    counts = adata.obs["Tier_A"].value_counts().rename("n").reset_index()
    counts.columns = ["Tier_A", "n"]
    counts["sample_id"] = cfg["sample_id"]
    counts["disease_group"] = cfg["disease_group"]
    summary_frames.append(counts)
    del adata
    gc.collect()

tier_a_counts_df = pd.concat(summary_frames, ignore_index=True)
tier_a_counts_df["fraction"] = tier_a_counts_df["n"] / tier_a_counts_df.groupby("sample_id")["n"].transform("sum")
save_df(tier_a_counts_df, OUTPUT_DIR / "xenium_tier_a_counts.csv")

cluster_annotation_summary_df = (
    pd.concat(cluster_summary_frames, ignore_index=True)
    if len(cluster_summary_frames) > 0
    else pd.DataFrame()
)
if len(marker_availability_rows) > 0:
    marker_availability_df = pd.DataFrame(marker_availability_rows)
    save_df(marker_availability_df, ANNOTATION_QC_DIR / "marker_gene_availability.csv")
else:
    marker_availability_path = ANNOTATION_QC_DIR / "marker_gene_availability.csv"
    marker_availability_df = pd.read_csv(marker_availability_path) if marker_availability_path.exists() else pd.DataFrame()

tier_a_counts_df.head()


## Export Xenium Explorer Cell Groups

These CSVs can be imported into Xenium Explorer as custom cell groups. `Tier_A` is best for broad QC; `Tier_B` is best when you want the refined subtype labels.


In [ ]:

XENIUM_EXPLORER_GROUP_DIR = OUTPUT_DIR / "xenium_explorer_cell_groups"
XENIUM_EXPLORER_GROUP_DIR.mkdir(exist_ok=True)

export_rows = []
for cfg in SAMPLE_CONFIGS:
    adata = sc.read_h5ad(ANNOTATED_DIR / f"{cfg['sample_id']}_annotated.h5ad", backed="r")
    for obs_col, suffix in {"Tier_A": "tier_a", "Tier_B": "tier_b"}.items():
        if obs_col not in adata.obs.columns:
            continue
        export_df = adata.obs[[obs_col]].copy()
        export_df.insert(0, "cell_id", export_df.index.astype(str))
        export_df = export_df.rename(columns={obs_col: "group"})
        export_df["group"] = export_df["group"].astype(str).fillna("Unknown")
        out_path = XENIUM_EXPLORER_GROUP_DIR / f"{cfg['sample_id']}_{suffix}_cell_groups.csv"
        export_df.to_csv(out_path, index=False)
        export_rows.append(
            {
                "sample_id": cfg["sample_id"],
                "annotation": obs_col,
                "n_cells": len(export_df),
                "path": str(out_path),
            }
        )
    adata.file.close()

xenium_explorer_export_df = pd.DataFrame(export_rows)
xenium_explorer_export_df


## Annotation Review Table

The table below is the key checkpoint. If any automated cluster label looks wrong, edit `final_Tier_A` and `final_Tier_B` in `xenium_cluster_annotation_review.csv`, set `FORCE_REANNOTATE = True`, and rerun the annotation cell. The notebook will preserve curated labels where `sample_id + leiden` still match.


In [ ]:

review_df = pd.read_csv(CLUSTER_REVIEW_PATH, dtype={"leiden": str})
review_df.head(20)


In [ ]:

if not marker_availability_df.empty:
    display(marker_availability_df.sort_values(["sample_id", "gene_set"]).head(30))
else:
    print("Marker availability table is empty because cached annotations were used before this table existed.")


### Epithelial Ambiguity Check

This table is specifically for the duodenum-vs-ductal issue. True duodenum should have strict intestinal anchors (`CDX2` plus `KRT20/VIL1/FABP/ALPI/MUC2` when available). In PDAC, `REG4`, `DMBT1`, `TFF3`, `TMPRSS2`, and `GPX2` are treated as intestinal-like/PanIN-like ductal remodeling unless those stricter intestinal anchors are also present.


In [ ]:

epithelial_qc_labels = ["pancreatic ductal epithelium", "Duodenum epithelial"]
epithelial_qc_cols = [
    "sample_id", "leiden", "n_cells", "fraction", "top_marker_genes",
    "suggested_Tier_A", "suggested_Tier_B", "annotation_note",
    "ductal_epithelial_score_z", "duodenum_epithelial_score_z",
    "intestinal_like_ductal_remodeling_score_z", "panin_mucin_remodeling_score_z",
    "EPCAM_frac_pos", "KRT7_frac_pos", "SOX9_frac_pos", "CFTR_frac_pos", "FXYD2_frac_pos",
    "CDX2_frac_pos", "REG4_frac_pos", "DMBT1_frac_pos", "TMPRSS2_frac_pos", "GPX2_frac_pos",
    "KRT20_frac_pos", "VIL1_frac_pos", "FABP1_frac_pos", "FABP2_frac_pos", "ALPI_frac_pos", "MUC2_frac_pos",
]
epithelial_qc_df = cluster_annotation_summary_df[
    cluster_annotation_summary_df["suggested_Tier_A"].isin(epithelial_qc_labels)
    | cluster_annotation_summary_df["suggested_Tier_B"].astype(str).str.contains("ductal|duodenum|intestinal", case=False, na=False)
].copy()
epithelial_qc_df = epithelial_qc_df[[c for c in epithelial_qc_cols if c in epithelial_qc_df.columns]]
display(epithelial_qc_df.sort_values(["sample_id", "suggested_Tier_A", "fraction"], ascending=[True, True, False]))
save_df(epithelial_qc_df, ANNOTATION_QC_DIR / "epithelial_ductal_duodenum_qc.csv")

refinement_paths = sorted(ANNOTATION_QC_DIR.glob("*_epithelial_refinement_qc.csv"))
if len(refinement_paths) > 0:
    epithelial_refinement_qc_df = pd.concat(
        [pd.read_csv(path) for path in refinement_paths],
        ignore_index=True,
    )
    display(
        epithelial_refinement_qc_df.sort_values(
            ["sample_id", "duodenum_minus_ductal_score"],
            ascending=[True, False],
        )
    )
else:
    print("No epithelial refinement QC files found. This is expected only if too few epithelial candidates are available.")


In [ ]:

plt.figure(figsize=(9, 4.5))
plot_df = tier_a_counts_df.sort_values(["sample_id", "fraction"], ascending=[True, False])
sns.barplot(
    data=plot_df,
    x="sample_id",
    y="fraction",
    hue="Tier_A",
    palette=TIER_A_PALETTE,
    linewidth=0,
)
plt.title("Marker-rule Tier_A composition")
plt.xlabel("")
plt.ylabel("Fraction of cells")
plt.xticks(rotation=25, ha="right")
plt.legend(frameon=False, fontsize=6, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(False)
plt.tight_layout()
plt.show()


## Cluster-Level Marker QC

These are the checks that should make wrong labels obvious: cluster score heatmaps, UMAPs by 10x graphclust / Leiden / Tier_A / Tier_B, marker dotplots by the annotation cluster key, and spatial maps.


In [ ]:

if cluster_annotation_summary_df.empty and CLUSTER_REVIEW_PATH.exists():
    cluster_annotation_summary_df = pd.read_csv(CLUSTER_REVIEW_PATH, dtype={"leiden": str})

score_cols = [c for c in cluster_annotation_summary_df.columns if c.endswith("_score_z")]
if len(score_cols) > 0:
    heatmap_df = cluster_annotation_summary_df.copy()
    if "cluster_key" not in heatmap_df.columns:
        heatmap_df["cluster_key"] = "cluster"
    heatmap_df["cluster_key"] = heatmap_df["cluster_key"].astype(str)
    heatmap_df["cluster"] = (
        heatmap_df["sample_id"].astype(str)
        + ":"
        + heatmap_df["cluster_key"].str.replace("xenium_", "", regex=False)
        + ":"
        + heatmap_df["leiden"].astype(str)
    )
    heatmap_df = heatmap_df.set_index("cluster")[score_cols]
    plt.figure(figsize=(max(7, 0.35 * len(score_cols)), max(4, 0.16 * len(heatmap_df))))
    sns.heatmap(
        heatmap_df,
        cmap="RdBu_r",
        center=0,
        linewidths=0,
        cbar_kws={"label": "Mean cluster marker-program z-score"},
    )
    plt.title("Cluster marker-program validation")
    plt.tight_layout()
    plt.show()
else:
    print("No score columns available for cluster-level heatmap.")


In [ ]:

def plot_downsampled_embedding(ax, obs, x, y, color, title, max_cells=50000, palette=None, cmap="viridis"):
    rng = np.random.default_rng(42)
    plot_obs = obs[[x, y, color]].dropna().copy()
    if len(plot_obs) > max_cells:
        plot_obs = plot_obs.iloc[rng.choice(len(plot_obs), size=max_cells, replace=False)]
    if pd.api.types.is_numeric_dtype(plot_obs[color]):
        sca = ax.scatter(plot_obs[x], plot_obs[y], c=plot_obs[color], cmap=cmap, s=1, alpha=0.65, linewidths=0)
        plt.colorbar(sca, ax=ax, fraction=0.046, pad=0.04)
    else:
        sns.scatterplot(
            data=plot_obs,
            x=x,
            y=y,
            hue=color,
            palette=palette,
            s=1,
            linewidth=0,
            alpha=0.65,
            ax=ax,
            legend=False,
        )
    ax.set_title(title)
    ax.grid(False)

for cfg in SAMPLE_CONFIGS:
    adata = sc.read_h5ad(ANNOTATED_DIR / f"{cfg['sample_id']}_annotated.h5ad")
    annotation_cluster_key = adata.uns.get("spatioev_annotation_cluster_key", None)
    if annotation_cluster_key is None or annotation_cluster_key not in adata.obs.columns:
        annotation_cluster_key = choose_annotation_cluster_key(adata)
    fig, axes = plt.subplots(1, 4, figsize=(15.5, 3.6))
    obs = adata.obs.join(pd.DataFrame(adata.obsm["X_umap"], index=adata.obs_names, columns=["UMAP1", "UMAP2"]))
    plot_downsampled_embedding(
        axes[0],
        obs,
        "UMAP1",
        "UMAP2",
        annotation_cluster_key,
        f"{cfg['sample_id']}: annotation clusters ({annotation_cluster_key})",
    )
    plot_downsampled_embedding(axes[1], obs, "UMAP1", "UMAP2", LEIDEN_KEY, f"{cfg['sample_id']}: Scanpy Leiden QC")
    plot_downsampled_embedding(
        axes[2],
        obs,
        "UMAP1",
        "UMAP2",
        "Tier_A",
        f"{cfg['sample_id']}: Tier_A",
        palette=TIER_A_PALETTE,
    )
    plot_downsampled_embedding(
        axes[3],
        obs,
        "UMAP1",
        "UMAP2",
        "Tier_B",
        f"{cfg['sample_id']}: Tier_B",
        palette=palette_for_values(obs["Tier_B"], TIER_B_BASE_PALETTE),
    )
    plt.tight_layout()
    plt.show()

    dotplot_markers = [g for g in DOTPLOT_MARKERS if g in adata.var_names]
    if len(dotplot_markers) > 0:
        sc.pl.dotplot(
            adata,
            var_names=dotplot_markers,
            groupby=annotation_cluster_key,
            layer="log1p",
            standard_scale="var",
            dendrogram=False,
            show=True,
            title=f"{cfg['sample_id']}: marker expression by annotation cluster ({annotation_cluster_key})",
        )
        sc.pl.dotplot(
            adata,
            var_names=dotplot_markers,
            groupby="Tier_A",
            layer="log1p",
            standard_scale="var",
            dendrogram=False,
            show=True,
            title=f"{cfg['sample_id']}: marker expression by final Tier_A annotation",
        )
        tier_b_counts = adata.obs["Tier_B"].value_counts()
        tier_b_keep = tier_b_counts.loc[tier_b_counts >= 100].index.tolist()
        adata_tier_b = adata[adata.obs["Tier_B"].isin(tier_b_keep)].copy()
        sc.pl.dotplot(
            adata_tier_b,
            var_names=dotplot_markers,
            groupby="Tier_B",
            layer="log1p",
            standard_scale="var",
            dendrogram=False,
            show=True,
            title=f"{cfg['sample_id']}: marker expression by final Tier_B annotation",
        )
        del adata_tier_b
    del adata
    gc.collect()


## Quick Spatial QC

These are lightweight downsampled plots to catch obvious annotation problems before building niches.


In [ ]:

MAX_PLOT_CELLS_PER_SAMPLE = 30000
rng = np.random.default_rng(42)

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
axes = axes.ravel()

for ax, cfg in zip(axes, SAMPLE_CONFIGS):
    adata = sc.read_h5ad(ANNOTATED_DIR / f"{cfg['sample_id']}_annotated.h5ad", backed="r")
    obs = adata.obs[["x_centroid", "y_centroid", "Tier_A", "disease_group"]].copy()
    if len(obs) > MAX_PLOT_CELLS_PER_SAMPLE:
        obs = obs.iloc[rng.choice(len(obs), size=MAX_PLOT_CELLS_PER_SAMPLE, replace=False)]
    sns.scatterplot(
        data=obs,
        x="x_centroid",
        y="y_centroid",
        hue="Tier_A",
        palette=TIER_A_PALETTE,
        s=1,
        linewidth=0,
        alpha=0.65,
        ax=ax,
        legend=False,
    )
    ax.set_title(cfg["sample_id"])
    ax.invert_yaxis()
    ax.set_aspect("equal", adjustable="box")
    ax.grid(False)
    adata.file.close()

plt.tight_layout()
plt.show()
